# 📈 Notebook 7 — Visualizations Gallery
**Formula 1 ML Analytics Project**

Publication-quality visualizations of all model outputs and insights.

| # | Chart | Type |
|---|-------|------|
| 1 | SHAP summary plot | Feature importance |
| 2 | SHAP force plot | Single prediction explanation |
| 3 | Predicted vs Actual | Model accuracy |
| 4 | GOAT radar chart | 5-axis comparison (Plotly) |
| 5 | Hypothetical season simulation | Animated points (Plotly) |
| 6 | Feature importance bar | Top 20 features |
| 7 | Confusion matrix | Win/no-win classification |
| 8 | Win probability by grid | Pearson r = 0.828 |
| 9 | Era-adjusted career trajectories | Interactive (Plotly) |
| 10 | Monte Carlo distribution | 1000-iteration histogram |
| 11 | Teammate comparison | GOAT H2H chart |
| 12 | Constructor dominance timeline | Historical eras |


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
DATA_PATH  = "../data/processed/"
MODEL_PATH = "../models/"
PLOTS_PATH = "../outputs/plots/"
os.makedirs(PLOTS_PATH, exist_ok=True)

# Load all available data
def try_load(path):
    if os.path.exists(path):
        return pd.read_csv(path, low_memory=False)
    print(f"⚠️  Not found: {path}")
    return None

master_df   = try_load(os.path.join(DATA_PATH, "master_df.csv"))
featured_df = try_load(os.path.join(DATA_PATH, "featured_df.csv"))
goat_df     = try_load(os.path.join(DATA_PATH, "goat_results.csv"))

# Load model
try:
    import joblib
    model_file = os.path.join(MODEL_PATH, "race_winner_model.pkl")
    race_model = joblib.load(model_file) if os.path.exists(model_file) else None
    print(f"Race model: {'✅ loaded' if race_model else '⚠️  not found'}")
except:
    race_model = None

from src.visualizations import *
print("\n✅ Visualizations module loaded.")
print(f"Saving plots to: {os.path.abspath(PLOTS_PATH)}")


In [ ]:
# ==========================================================
# Plot 1 + 2: SHAP Summary & Force Plot
# ==========================================================
if race_model is not None and featured_df is not None:
    from src.models import prepare_features, get_time_split, FEATURE_COLS
    try:
        import shap
        _, _, test_df = get_time_split(featured_df)
        X_test, _, _ = prepare_features(test_df)
        
        explainer = shap.TreeExplainer(race_model)
        shap_vals  = explainer.shap_values(X_test)
        
        # Plot 1 — Summary
        plot_shap_summary(shap_vals, X_test, save_path=os.path.join(PLOTS_PATH, "01_shap_summary.png"))
        
        # Plot 2 — Force (single prediction)
        plot_shap_force(explainer, X_test.iloc[:1], save_path=os.path.join(PLOTS_PATH, "02_shap_force.png"))
    except Exception as e:
        print(f"SHAP plots skipped: {e}")
else:
    print("⚠️  SHAP plots require trained model and featured_df. Run notebooks 01–03 first.")


In [ ]:
# ==========================================================
# Plot 3: Predicted vs Actual
# ==========================================================
if race_model is not None and featured_df is not None:
    from src.models import prepare_features, get_time_split
    try:
        _, _, test_df = get_time_split(featured_df)
        X_test, y_win, y_pos = prepare_features(test_df)
        y_pred_pos = race_model.predict(X_test)
        plot_predicted_vs_actual(y_pos, y_pred_pos,
                                 save_path=os.path.join(PLOTS_PATH, "03_pred_vs_actual.png"))
    except Exception as e:
        print(f"Plot 3 skipped: {e}")
else:
    print("⚠️  Plot 3 requires trained model. Run notebook 03 first.")


In [ ]:
# ==========================================================
# Plot 4: GOAT Radar Chart (Plotly)
# ==========================================================
if goat_df is not None:
    plot_goat_radar(goat_df, save_path=os.path.join(PLOTS_PATH, "04_goat_radar.html"))
else:
    print("⚠️  GOAT radar requires goat_results.csv. Run notebook 06 first.")


In [ ]:
# ==========================================================
# Plot 5: Hypothetical Season Simulation (Plotly)
# ==========================================================
if featured_df is not None:
    from src.hypothetical_engine import hypothetical_scenario
    try:
        sim_result = hypothetical_scenario(
            featured_df=featured_df,
            driver="Sebastian Vettel",
            team="Alfa Romeo",
            year=2022,
            reliability_factor=0.88,
            race_incidents=True,
            n_monte_carlo=100,
            model=race_model
        )
        plot_hypothetical_season(sim_result, 
                                  save_path=os.path.join(PLOTS_PATH, "05_hypothetical_season.html"))
    except Exception as e:
        print(f"Plot 5 skipped: {e}")


In [ ]:
# ==========================================================
# Plot 6: Feature Importance Bar Chart
# ==========================================================
if race_model is not None and featured_df is not None:
    from src.models import FEATURE_COLS
    feature_names = [c for c in FEATURE_COLS if c in featured_df.columns]
    try:
        plot_feature_importance(race_model, feature_names, top_n=20,
                               save_path=os.path.join(PLOTS_PATH, "06_feature_importance.png"))
    except Exception as e:
        print(f"Plot 6 skipped: {e}")
else:
    print("⚠️  Feature importance requires trained model.")


In [ ]:
# ==========================================================
# Plot 7: Confusion Matrix
# ==========================================================
if race_model is not None and featured_df is not None:
    from src.models import prepare_features, get_time_split
    try:
        _, _, test_df = get_time_split(featured_df)
        X_test, y_win, _ = prepare_features(test_df)
        y_pred_win = race_model.predict(X_test)
        y_pred_bin = (y_pred_win > 0.5).astype(int) if y_pred_win.dtype == float else y_pred_win
        plot_confusion_matrix(y_win, y_pred_bin,
                              save_path=os.path.join(PLOTS_PATH, "07_confusion_matrix.png"))
    except Exception as e:
        print(f"Plot 7 skipped: {e}")


In [ ]:
# ==========================================================
# Plot 8: Win Probability by Grid Position
# ==========================================================
if featured_df is not None:
    plot_win_prob_by_grid(featured_df, 
                          save_path=os.path.join(PLOTS_PATH, "08_win_prob_by_grid.png"))
else:
    print("⚠️  Plot 8 requires featured_df.")


In [ ]:
# ==========================================================
# Plot 9: Era-Adjusted Career Trajectories (Plotly)
# ==========================================================
if master_df is not None:
    plot_career_trajectories(
        master_df,
        drivers=['Lewis Hamilton','Michael Schumacher','Max Verstappen',
                 'Sebastian Vettel','Alain Prost','Ayrton Senna'],
        save_path=os.path.join(PLOTS_PATH, "09_career_trajectories.html")
    )
else:
    print("⚠️  Plot 9 requires master_df.")


In [ ]:
# ==========================================================
# Plot 10: Monte Carlo Distribution
# ==========================================================
if featured_df is not None:
    from src.hypothetical_engine import hypothetical_scenario
    try:
        mc_result = hypothetical_scenario(
            featured_df=featured_df,
            driver="Lewis Hamilton",
            team="Haas F1 Team",
            year=2023,
            reliability_factor=0.85,
            race_incidents=True,
            n_monte_carlo=1000,
            model=race_model
        )
        mc_pts = mc_result.get('monte_carlo_results', [])
        if mc_pts:
            plot_monte_carlo_distribution(mc_pts, driver="Lewis Hamilton", 
                                          team="Haas F1 Team", year=2023,
                                          save_path=os.path.join(PLOTS_PATH, "10_monte_carlo.png"))
    except Exception as e:
        print(f"Plot 10 skipped: {e}")


In [ ]:
# ==========================================================
# Plot 11: Teammate Comparison
# ==========================================================
if goat_df is not None:
    plot_teammate_comparison(goat_df, 
                              save_path=os.path.join(PLOTS_PATH, "11_teammate_comparison.png"))
elif featured_df is not None:
    from src.goat_analysis import compute_teammate_comparison
    CANDIDATES = ['Lewis Hamilton','Michael Schumacher','Max Verstappen','Sebastian Vettel',
                  'Alain Prost','Ayrton Senna','Fernando Alonso']
    tm_df = compute_teammate_comparison(featured_df, featured_df, candidates=CANDIDATES)
    plot_teammate_comparison(tm_df, 
                              save_path=os.path.join(PLOTS_PATH, "11_teammate_comparison.png"))


In [ ]:
# ==========================================================
# Plot 12: Constructor Dominance Timeline
# ==========================================================
if master_df is not None:
    plot_constructor_dominance(master_df,
                               save_path=os.path.join(PLOTS_PATH, "12_constructor_dominance.png"))
elif featured_df is not None:
    plot_constructor_dominance(featured_df,
                               save_path=os.path.join(PLOTS_PATH, "12_constructor_dominance.png"))
else:
    print("⚠️  Plot 12 requires master_df.")


In [ ]:
# Summary of all generated plots
import glob
plots = glob.glob(os.path.join(PLOTS_PATH, "*"))
print(f"\n✅ Generated {len(plots)} visualization files in {os.path.abspath(PLOTS_PATH)}:")
for p in sorted(plots):
    size_kb = os.path.getsize(p) / 1024
    print(f"  📊 {os.path.basename(p)} ({size_kb:.0f} KB)")
